In [25]:
import numpy as np 
import pandas as pd 
from mosqlient.scoring import compute_wis

In [3]:
challenge = 'dengue_state'

df_preds = pd.read_csv(f'../predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,wis,model
0,2022-10-09,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
1,2022-10-16,37.019327,43.295517,51.743385,66.653189,83.207109,100.790847,116.992117,126.864602,134.406880,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
2,2022-10-23,55.945220,68.411390,84.200302,108.887826,135.698419,163.344639,189.017119,205.765306,218.575663,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
3,2022-10-30,58.185781,69.697014,83.792687,107.076328,132.601767,158.745145,184.276095,201.204977,212.912726,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
4,2022-11-06,58.634062,68.591352,80.855678,101.969117,125.548661,150.259612,174.517426,190.524410,202.512661,12,6822,1,95.67,3rd_imdc_isi_isi-dengue


In [15]:
def get_data(name = 'dengue_state'): 

    df = pd.read_csv(f'../data/{name}.csv.gz')

    return df

In [16]:
df_dengue_state = get_data('dengue_state')


### RK 1: 

In [26]:
def compute_metrics(model, df_w, df_preds_, adm_1=None):
    '''
    Function to compute the score for the entire validation test or only around the peak (if peak=True).
    '''

    df_preds_2 = df_preds_
    df_w2 = df_w
        
    df_preds_model = df_preds_2.loc[df_preds_2.model == model].reset_index(drop = True)
        
    df_preds_to_score = df_w2.merge(df_preds_model, left_on = ['date', 'adm_1'], right_on = ['date', 'adm_1'])

    wis = np.mean(compute_wis( 
                        df_preds_to_score[['date',  'lower_95', 'lower_90', 'lower_80', 'lower_50',
                           'pred', 'upper_50', 'upper_80', 'upper_90', 'upper_95']],
                        observed_value = df_preds_to_score['casos'].values)) 
    return wis

In [27]:
df_metrics = pd.DataFrame()

for model in df_preds.model.unique(): 

    for state in df_preds.adm_1.unique():

        if state != 32:  

            wis = compute_metrics(model, df_dengue_state.loc[df_dengue_state.adm_1 == state],
                        df_preds.loc[df_preds.adm_1 == state])
            
            df_metrics = pd.concat([df_metrics,
                                pd.DataFrame([[model, state, 'all', wis ]],
                                    columns = ['model', 'adm_1', 'validation_test', 'WIS'] 
                                    )], ignore_index = True)
            

df_metrics['rank'] = df_metrics.groupby('adm_1')['WIS'].rank(method='first').astype(int)

rank_wis = df_metrics.sort_values('rank')        

rank_wis

,model,adm_1,validation_test,WIS,rank
692,3rd_imdc_emap_epidematicos_sarimax_state,14,all,3.033120,1
695,3rd_imdc_emap_epidematicos_sarimax_state,28,all,11.451357,1
677,3rd_imdc_emap_epidematicos_sarimax_state,27,all,51.189035,1
341,3rd_imdc_pucrio_arbocaster,13,all,25.431297,1
351,3rd_imdc_pucrio_arbocaster,24,all,56.693063,1
...,...,...,...,...,...
501,3rd_imdc_fiocruz_zerolags,51,all,525.811965,27
52,3rd_imdc_fiocruz_mard,12,all,172.497588,27
375,3rd_imdc_emap_lstm,50,all,401.473086,27
340,3rd_imdc_pucrio_arbocaster,16,all,156.821536,27


In [28]:
rank_wis.to_csv('default_rk_1.csv.gz', index = False)

### RK 2: 

In [19]:
df_dengue_state["date"] = pd.to_datetime(df_dengue_state["date"])

df_total_cases = (
    df_dengue_state.merge(
        df_preds[["date", "adm_1", "validation"]].drop_duplicates(
                subset=["date", "adm_1", "validation"],
                keep="first"
            ), on=["date", "adm_1"]
    )
    .groupby(["adm_1", "validation"])[["casos"]]
    .sum()
    .reset_index()
)

df_new_wis = df_total_cases.merge(
    df_preds[["model", "adm_1", "validation", "wis"]].drop_duplicates(
                subset=["model", "adm_1", "validation", "wis"],
                keep="first"
            ), on=["adm_1", "validation"]
)

condicoes = [df_new_wis["validation"].isin([1, 2, 3]), df_new_wis["validation"] == 4]
multiplicadores = [52, 53]

df_new_wis["wis_norm"] = (
    np.select(condicoes, multiplicadores, default=np.nan) * df_new_wis["wis"]
) / df_new_wis["casos"]

df_new_wis.head()

,adm_1,validation,casos,model,wis,wis_norm
0,11,1,13281,3rd_imdc_isi_isi-dengue,239.780000,0.938827
1,11,1,13281,3rd_imdc_purdue_neuralearth,93.159748,0.364755
2,11,1,13281,3rd_imdc_fiocruz_mard,68.696713,0.268973
3,11,1,13281,3rd_imdc_bsc_ghr,127.461645,0.499059
4,11,1,13281,3rd_imdc_fgv_pattern-blue,167.210087,0.654689


In [22]:
m_wis_norm = (
    df_new_wis
    .groupby(["adm_1", "model"], as_index=False)["wis_norm"]
    .mean()
)

m_wis_norm["rank"] = (
    m_wis_norm
    .groupby("adm_1")["wis_norm"]
    .rank(method="first", ascending=True)
    .astype(int)
)

rank_wis_norm = (
    m_wis_norm
    .sort_values(["adm_1", "rank"])
)

rank_wis_norm.head()

,adm_1,model,wis_norm,rank
6,11,3rd_imdc_emap_epidematicos_sarimax_state,0.407169,1
20,11,3rd_imdc_pucrio_arbocaster,0.418089,2
13,11,3rd_imdc_ifgw_inframind-proteus,0.424080,3
16,11,3rd_imdc_lncc_lncc_arp26_dengue,0.424529,4
23,11,3rd_imdc_rki_rki_zki_ph_lstm_geo,0.455836,5


In [23]:
rank_wis_norm.to_csv('norm_rk_2.csv.gz', index = False)

## RK 4: Gerando um ranking pela média dos rankings em cada conjunto: 

In [17]:
df_agg_wis = (
    df_preds
    .groupby(["model", "adm_1", "validation"], as_index=False)["wis"]
    .mean()
)


In [18]:
# Rank dentro de cada estado e conjunto de validação
df_rank = df_agg_wis.copy()

df_rank["rank"] = (
    df_rank
    .groupby(["adm_1", "validation"])["wis"]
    .rank(method="average", ascending=True)
)

# Rank médio por modelo e estado
df_mean_rank = (
    df_rank
    .groupby(["adm_1", "model"], as_index=False)["rank"]
    .mean()
    .rename(columns={"rank": "mean_rank"})
    .sort_values(["adm_1", "mean_rank"])
)

df_mean_rank['rank'] = df_mean_rank.groupby(['adm_1'])['mean_rank'].rank(method= 'min', ascending = True)

df_mean_rank.head()

,adm_1,model,mean_rank,rank
6,11,3rd_imdc_emap_epidematicos_sarimax_state,6.25,1.0
16,11,3rd_imdc_lncc_lncc_arp26_dengue,6.75,2.0
20,11,3rd_imdc_pucrio_arbocaster,7.25,3.0
22,11,3rd_imdc_rki_rki_zki_ph,8.25,4.0
21,11,3rd_imdc_purdue_neuralearth,8.50,5.0


In [21]:
df_mean_rank.to_csv('mean_rk_4.csv.gz', index = False)

Gerando um ranking da média da diferença entre os modelos e o baseline: 

In [13]:
model_baseline = '3rd_imdc_procc_bb_model'

df_baseline = (
    df_agg_wis.loc[df_agg_wis["model"] == model_baseline,
           ["adm_1", "validation", "wis"]]
    .rename(columns={"wis": "wis_baseline"})
)

# Junta o WIS do baseline aos demais modelos
df_ratio = df_agg_wis.merge(
    df_baseline,
    on=["adm_1", "validation"],
    how="left"
)

# Razão WIS / Baseline
df_ratio["wis_ratio"] = (
    df_ratio["wis"] / df_ratio["wis_baseline"]
)

# Médias das razões
df_summary = (
    df_ratio
    .groupby(["adm_1", "model"], as_index=False)
    .agg(
        arithmetic_mean_ratio=("wis_ratio", "mean"),
        geometric_mean_ratio=("wis_ratio", lambda x: np.exp(np.mean(np.log(x)))),
    )
    .sort_values(["adm_1", "geometric_mean_ratio"])
)


df_summary['rank'] = df_summary.groupby(['adm_1'])['geometric_mean_ratio'].rank(method= 'min', ascending = True)


df_summary.head()

,adm_1,model,arithmetic_mean_ratio,geometric_mean_ratio,rank
6,11,3rd_imdc_emap_epidematicos_sarimax_state,0.803841,0.769598,1.0
20,11,3rd_imdc_pucrio_arbocaster,0.834013,0.789304,2.0
16,11,3rd_imdc_lncc_lncc_arp26_dengue,0.818201,0.800598,3.0
13,11,3rd_imdc_ifgw_inframind-proteus,0.964784,0.845450,4.0
22,11,3rd_imdc_rki_rki_zki_ph,0.914512,0.893327,5.0


In [14]:
#df_ratio['region'] = df_ratio['adm_1'].replace(code_to_state).replace(estado_para_regiao)

df_summary.to_csv('diff_rk_3.csv.gz', index = False)